In [6]:
import pandas as pd
import numpy as np
import re
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

# 1. Load and preprocess data
df = pd.read_csv("Datasets/CBOW/IMDB Dataset.csv")

stop_words = set(ENGLISH_STOP_WORDS)

def preprocess(text):
    words = re.findall(r'\b[a-z]+\b', text.lower())
    return [w for w in words if w not in stop_words]

sentences = df["review"].fillna("").apply(preprocess).tolist()

# 2. Encode labels
le = LabelEncoder()
y = le.fit_transform(df["sentiment"])

# 3. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    sentences, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Train Word2Vec using CBOW
w2v = Word2Vec(X_train, vector_size=100, window=5,
               min_count=2, sg=0, workers=4, epochs=10)

# 5. Convert documents into average word embeddings
def document_vector(words):
    vectors = [w2v.wv[w] for w in words if w in w2v.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

X_train = np.array([document_vector(s) for s in X_train])
X_test = np.array([document_vector(s) for s in X_test])

print("First 5 document embeddings:")
print(X_train[:5])

# 6. Build feed forward neural network
model = Sequential([
    Input(shape=(100,)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])

# 7. Train the model
model.fit(X_train, y_train, epochs=5, batch_size=32,
          validation_split=0.1)

# 8. Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

#  Sample test
review = "The movie was amazing and the acting was excellent"

words = preprocess(review)
embedding = document_vector(words)

print("Review:", review)
print("Embedding:", embedding)

pred = model.predict(embedding.reshape(1, 100), verbose=0)[0][0]

print("Predicted Sentiment:",
      "Positive" if pred >= 0.5 else "Negative")

First 5 document embeddings:
[[ 0.19373842 -0.00856654 -0.24800213 -0.07680085  0.3437828  -0.14775686
   0.33219123  0.2058597  -0.11808417  0.2968668  -0.10425707 -0.0995422
  -0.2376653   0.17881842 -0.03378893  0.29670715 -0.31959066 -0.03342996
   0.10107345 -0.6964522   0.23038468  0.0691186  -0.0084345  -0.64378154
  -0.16795085 -0.04613529 -0.07756319 -0.19711699  0.08171237  0.24337342
   0.03688513 -0.0612732  -0.10187107  0.0583697  -0.5415595  -0.5204029
   0.5145704   0.44852996 -0.16716051 -0.2710657  -0.30440012  0.2860049
  -0.03377035 -0.2896369   0.34765902  0.4476725  -0.30804464  0.24659997
  -0.14270644  0.5056838  -0.6518155  -0.12804861  0.04143151 -0.15050916
  -0.2348482  -0.0555866  -0.0074767  -0.4604582  -0.10944595  0.15943028
   0.41847232  0.1569033   0.36270887 -0.56112844 -0.13485053  0.3638762
  -0.32747686  0.16618004 -0.03870431 -0.6496419  -0.02525699 -0.03054615
  -0.15232587  0.28805122 -0.13609633  0.06756491  0.68717456  0.53790724
  -0.2074161 

In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Sequential

df = pd.read_csv("Datasets/CBOW/IMDB Dataset.csv")

stop_words = set(ENGLISH_STOP_WORDS)

def preprocess(text):
    words = re.findall(r'\b[a-z]+\b', text.lower())
    return [w for w in words if w not in stop_words]

sentences = df["review"].fillna("").apply(preprocess).tolist()

encoder = LabelEncoder()
y = encoder.fit_transform(df["sentiment"])

x_train, x_test, y_train, y_test = train_test_split(sentences, y, test_size=0.2, stratify=y)

w2v = Word2Vec(x_train, vector_size=100, sg=0, min_count=2, window=5, workers=4, epochs=10)

def document_vector(words):
    vectors = [w2v.wv[w] for w in words if w in w2v.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

x_train = np.array([document_vector(s) for s in x_train])
x_test = np.array([document_vector(s) for s in x_test])

model = Sequential([
    Input(shape=(100,)),
    Dense(64, activation='relu'),
    Dense(28, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam', loss='binary_crossentropy', metrics=["accuracy"]
)

model.fit(x_train, y_train, epochs=5, batch_size=32, validation_split=0.1)

loss, accuracy = model.evaluate(x_test, y_test)
print("Loss: ", loss)
print("Accuracy: ", accuracy)

review = "fucking bull shit movie, ass shit"

words = preprocess(review)
embedding = document_vector(words)

print(review)
print(embedding)

pred = model.predict(embedding.reshape(1, 100), verbose=0)[0][0]

print("Positive" if pred >= 0.5 else "Negative")

Epoch 1/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8269 - loss: 0.3930 - val_accuracy: 0.8652 - val_loss: 0.3184
Epoch 2/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8606 - loss: 0.3245 - val_accuracy: 0.8605 - val_loss: 0.3173
Epoch 3/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8703 - loss: 0.3081 - val_accuracy: 0.8608 - val_loss: 0.3190
Epoch 4/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8682 - loss: 0.3103 - val_accuracy: 0.8627 - val_loss: 0.3180
Epoch 5/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8752 - loss: 0.3003 - val_accuracy: 0.8702 - val_loss: 0.3104
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8716 - loss: 0.3139
Loss:  0.313997358083725
Accuracy:  0.8701000213623047
fucking bull shit movie, ass shit
[ 0.47003493  0.7165015   0.12384833  0.13426146  0.42250738 -0.07923438
  0.43343535 -0.9832196  -0.20138037 -0.31118992 -0.37070498  0.17662844
  0.355825   -0.6905357   0.4279707  -0.03549